In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../..'))
sys.path.append(os.path.abspath('..'))

from loader import load_vae_training_log, load_specified_vae_training_log


from controller.marl.core.config import GenerativeLangType

from project_paths import FIGURES_DIR

from plt_style import set_style
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

set_style()

In [ ]:
# VQ-VAE
# TZs = [
#     "2026-04-16_13-39-36",
#     "2026-04-16_13-52-14",
#     "2026-04-16_14-05-02",
# ]

# VQ-VAE Non normalised
TZs = [
    "2026-04-16_15-47-58",
    "2026-04-16_16-10-40",
    "2026-04-16_16-34-01"
]

# VQ-VAE No K means
# TZs = [
#     "2026-04-16_14-43-16",
#     "2026-04-16_14-56-37",
#     "2026-04-16_15-11-38",
# ]

In [ ]:
# df = load_vae_training_log(GenerativeLangType.VQ_VAE, most_recent=3)
df = load_specified_vae_training_log(TZs)

In [ ]:
n_lang = len(TZs)
n_rows = 50

key = "validation_reconstruction_loss"

start_ixs = np.arange(0, n_lang * n_rows, n_rows)
final_ixs = start_ixs + n_rows - 1

final_values = df.iloc[final_ixs][key].values
mean = final_values.mean()
std  = final_values.std()

print("Final values:", final_values)
print("Mean:", mean, "Std:", std)

In [ ]:
pairs = [
    ("train_reconstruction_loss", "validation_reconstruction_loss", "Reconstruction Loss", "recon_loss_eval.png"),
    ("train_commitment_loss_0", "validation_commitment_loss_0", "Commitment Loss", "commit_loss_eval.png"),
]

for train_key, val_key, main_label, filename in pairs:
    fig, ax = plt.subplots(figsize=(9, 5))

    sns.lineplot(
        data=df,
        x="timestep",
        y=train_key,
        ax=ax,
        errorbar="ci",
        label="Training",
        color="tab:blue",
    )

    sns.lineplot(
        data=df,
        x="timestep",
        y=val_key,
        ax=ax,
        errorbar="ci",
        label="Validation",
        color="tab:orange",
    )

    plt.title(f"Average {main_label} During Training vs Validation Across All Logs (95% CI)")
    plt.ylabel(main_label)
    plt.xlabel("Timestep")
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close('all')